# Project 56 — Local CLI Command Planner Agent
## Describe a Task → Get Commands with Safety Checks

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic

## Step 1 — Setup Risk-Aware Command Generator

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

class CLIStep(BaseModel):
    command: str
    explanation: str
    risk_level: str = Field(description="safe, moderate, dangerous")
    reversible: bool
    platform: str = Field(description="linux, windows, macos, cross-platform")

class CLIPlan(BaseModel):
    task: str
    steps: list[CLIStep]
    warnings: list[str]
    prerequisites: list[str]
    estimated_duration: str

planner = llm.with_structured_output(CLIPlan)
print("CLI planner ready!")

## Step 2 — Generate & Review Plans

In [ ]:
tasks = [
    "Find all Python files larger than 1MB in the project",
    "Set up a Python virtual environment and install requirements",
    "Find and kill a process using port 8080",
    "Create a compressed backup of the src/ directory",
    "Search for 'TODO' comments across all source files",
    "Check disk space usage per directory",
]

for task in tasks:
    print(f"\nTask: {task}")
    print("-" * 50)
    plan = planner.invoke(f"Create a CLI plan for: {task}")
    if plan.prerequisites:
        print(f"  Prerequisites: {plan.prerequisites}")
    if plan.warnings:
        print(f"  ⚠ Warnings: {plan.warnings}")
    for i, step in enumerate(plan.steps, 1):
        risk_icon = {"safe": "🟢", "moderate": "🟡", "dangerous": "🔴"}[step.risk_level]
        rev = "↩" if step.reversible else "⚠"
        print(f"  {i}. {risk_icon} [{step.platform}] {step.command}")
        print(f"     {step.explanation} {rev}")
    print(f"  Duration: {plan.estimated_duration}")

## Step 3 — Safety Gate

In [ ]:
class SafetyCheck(BaseModel):
    is_safe: bool
    risk_category: str = Field(description="data_loss, system_change, network, none")
    explanation: str
    safer_alternative: str = Field(default="")

safety = llm.with_structured_output(SafetyCheck)

dangerous_commands = [
    "rm -rf /tmp/*",
    "chmod 777 /etc/passwd",
    "DROP TABLE users;",
    "kill -9 $(pgrep python)",
    "ls -la /home",
]

print("SAFETY GATE")
print("=" * 50)
for cmd in dangerous_commands:
    check = safety.invoke(f"Evaluate the safety of this command: {cmd}")
    icon = "✓" if check.is_safe else "✗"
    print(f"  {icon} {cmd}")
    print(f"    Risk: {check.risk_category} — {check.explanation[:80]}")
    if check.safer_alternative:
        print(f"    Safe alt: {check.safer_alternative}")
    print()

## Step 4 — Interactive Workflow Builder

In [ ]:
class Workflow(BaseModel):
    name: str
    description: str
    steps: list[CLIStep]
    environment_variables: list[str]
    shell_script: str = Field(description="Combined shell script")

workflow_gen = llm.with_structured_output(Workflow)

workflow = workflow_gen.invoke(
    "Create a CI/CD deployment workflow that: "
    "1) runs tests, 2) builds a Docker image, 3) pushes to registry, "
    "4) deploys to staging. Include error handling."
)

print(f"Workflow: {workflow.name}")
print(f"Description: {workflow.description}")
print(f"\nSteps:")
for i, s in enumerate(workflow.steps, 1):
    print(f"  {i}. {s.command}")
print(f"\nEnv vars needed: {workflow.environment_variables}")
print(f"\nGenerated script:\n{workflow.shell_script[:500]}")

## What You Learned
- **Risk-aware CLI planning** with safety levels
- **Safety gate** that blocks dangerous commands
- **Workflow generation** with shell scripts
- **Structured output** for actionable CLI plans